In [1]:
!pip install --upgrade gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 46.1 MB/s eta 0:00:00


In [2]:
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 13.5 MB/s eta 0:00:00


In [3]:
# =============================================================================
# MILESTONE 2 - HATEBR TEXT CLASSIFICATION (Simplified)
# Dataset: HateBR[](https://huggingface.co/datasets/franciellevargas/HateBR)
# Baselines:
#   A) TF-IDF (1-2 n-grams) + linguistic features + LogisticRegression
#   B) GloVe Average Embeddings + linguistic features + RandomForestClassifier
# =============================================================================

import pandas as pd
import numpy as np
import re
import spacy
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix
from gensim.models import KeyedVectors
import emoji
import warnings
warnings.filterwarnings('ignore')

1. LOADING THE DATASET (as in Project 1)

In [4]:
# ============================================================
# CARGAR DATASET DESDE GITHUB (como en Proyecto 1)
# ============================================================

# URL "raw" del archivo HateBR.csv en GitHub
url = "https://raw.githubusercontent.com/franfgv9/Milestone_1_PLN/refs/heads/main/HateBR.csv"

# Leer el CSV directamente desde GitHub
df = pd.read_csv(url)

# Verificar la carga
print("Dataset dimensions:", df.shape)
print("Available columns:", df.columns.tolist())
df.head()

Dataset dimensions: (7000, 8)
Available columns: ['id', 'comentario', 'anotator1', 'anotator2', 'anotator3', 'label_final', 'links_post', 'account_post']


,id,comentario,anotator1,anotator2,anotator3,label_final,links_post,account_post
0,1,Mais um lixo,1,1,1,1,https://www.instagram.com/p/B2uThqdH9xI/,Carla Zambelli
1,2,Essa nao tem vergonha na cara!!,1,1,1,1,https://www.instagram.com/p/B2uThqdH9xI/,Carla Zambelli
2,3,Essa mulher é doente.pilantra!,1,1,1,1,https://www.instagram.com/p/B2uThqdH9xI/,Carla Zambelli
3,4,Comunista safada...,1,1,1,1,https://www.instagram.com/p/B2uThqdH9xI/,Carla Zambelli
4,5,Vagabunda. Comunista. Mentirosa. O povo chilen...,1,1,1,1,https://www.instagram.com/p/B2uThqdH9xI/,Carla Zambelli


In [5]:
# ============================================================
# SELECCIONAR COLUMNAS RELEVANTES
# ============================================================

df = df[["id", "comentario", "label_final"]]

# Verificar el resultado
print("\nSelected columns:", df.columns.tolist())
print("Dataset dimensions:", df.shape)
df.head()


Selected columns: ['id', 'comentario', 'label_final']
Dataset dimensions: (7000, 3)


,id,comentario,label_final
0,1,Mais um lixo,1
1,2,Essa nao tem vergonha na cara!!,1
2,3,Essa mulher é doente.pilantra!,1
3,4,Comunista safada...,1
4,5,Vagabunda. Comunista. Mentirosa. O povo chilen...,1


In [6]:
# ============================================================
# RENOMBRAR COLUMNAS PARA UNIFORMIDAD
# ============================================================

df = df.rename(columns={
    "comentario": "text",
    "label_final": "label"
})

print("\nFinal columns:", df.columns.tolist())
df.head()

print("\nDataset information:")
print(df.info())

# Limpiar valores nulos
df.dropna(subset=['text', 'label'], inplace=True)
df = df[df['label'].isin([0, 1])]  # 0: No ofensivo, 1: Ofensivo
df.reset_index(drop=True, inplace=True)

print(f"\nFinal dataset: {len(df)} ejemplos")
print("Class distribution:")
print(df['label'].value_counts())


Final columns: ['id', 'text', 'label']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      7000 non-null   int64 
 1   text    7000 non-null   object
 2   label   7000 non-null   int64 
dtypes: int64(2), object(1)
memory usage: 164.2+ KB
None

Final dataset: 7000 ejemplos
Class distribution:
label
1    3500
0    3500
Name: count, dtype: int64


2. GENERAL CLEANING (for linguistic features)

The idea is:

Lightly clean up the text, but without “destroying” it, so that you can later extract linguistic features (POS tags, entities, dependencies, etc.).

Save that clean version in a new column of the DataFrame.

Download and load spaCy in Portuguese, which you will use later to extract those features.

In [7]:
# ============================================================
# PREPROCESAMIENTO GENERAL (para features lingüísticas)
# ============================================================

def clean_for_features(text):
    text = re.sub(r'http[s]?://\S+', '', text)  # Quitar URLs
    text = re.sub(r'\s+', ' ', text).strip()    # Normalizar espacios
    return text

df['text_features'] = df['text'].apply(clean_for_features)

# Descargar modelo spaCy para portugués si no está instalado
!python -m spacy download pt_core_news_sm

# Cargar modelo spaCy para portugués
print("\nLoading spaCy model (pt_core_news_sm)...")
nlp = spacy.load("pt_core_news_sm")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 88.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

Loading spaCy model (pt_core_news_sm)...


Why not clean it up more?
→ Because we want to preserve capital letters, symbols, and emojis to extract features.
We only remove URLs and extra spaces → they do not affect linguistic analysis.

In [8]:
# Aquí usas expresiones regulares con re.sub para eliminar enlaces.
  # re.sub(patrón, reemplazo, texto)
  # → busca el patrón en texto y lo sustituye por reemplazo.
  # Desglose del patrón r'http[s]?://\S+':
  # r'': raw string → para que Python no interprete \ como escapes raros.
  # 'http': busca exactamente la secuencia de letras http.
  # '[s]?':
  # [s] significa la letra s.
  # ? significa “0 o 1 vez”.
  # Es decir, coincide con http:// y también con https://.
  # '://': los caracteres :// literalmente.
  # \S+:
  # \S = “cualquier carácter que no sea espacio en blanco”.
  # + = “1 o más veces”.

In [9]:
# Patrón r'\s+':
  # \s = cualquier espacio en blanco (espacio, tabulación, salto de línea, etc.).
  # + = uno o más seguidos.
  # Es decir, detecta “bloques de espacios en blanco”.
  # re.sub(r'\s+', ' ', text):
  # Sustituye cualquier bloque de espacios/blancos (por ejemplo, " ", "\n\n", "\t \n") por un único espacio " ".
  # Esto “normaliza” la separación entre palabras.
  # .strip():
  # Elimina espacios al principio y al final del string.
  # Así no quedan frases empezando o terminando con espacios.
  # Ejemplo:
    # Antes: " Olha isso: ridículo \n\n "
    # Después: "Olha isso: ridículo"

3. SPECIFIC CLEANING BY MODEL

In [10]:
# ============================================================
# LIMPIEZA ESPECÍFICA POR MODELO
# ============================================================

# --- Modelo A: TF-IDF (superficial) ---   (es el mismo que hacemos antes de forma genérica)
def clean_for_tfidf(text):
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text_clean_tfidf'] = df['text'].apply(clean_for_tfidf)

# No se baja a minúsculas → lowercase=False en TfidfVectorizer
# Se preservan mayúsculas → "IDIOTA" ≠ "idiota" → señal de ofensa

# --- Modelo B: Embeddings (fuerte) ---
emoji_map = {'[laughing face]': 'risada', '[angry face]': 'raiva', '[heart]': 'amor'}

def replace_emojis(text):
    for emj, word in emoji_map.items():
        text = text.replace(emj, f' {word} ')
    return text

def clean_for_embeddings(text):
    text = re.sub(r'http[s]?://\S+', '', text)
    text = replace_emojis(text)
    text = text.lower()
    text = re.sub(r'[^a-záéíóúâêôãõç\s]', ' ', text)     # todo lo que no sean vocales acentuadas o letras en minúscula a-z se quita: GloVe no entiende símbolos, números, puntuación extraña
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Baja a minúsculas → GloVe es case-sensitive
# Reemplaza emojis → "risada" en lugar de cara riendo
# Solo letras y acentos → compatibilidad con GloVe

df['text_clean_emb'] = df['text'].apply(clean_for_embeddings)

4. EXTRACTION OF LINGUISTIC FEATURES

In [11]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

portuguese_stopwords = stopwords.words('portuguese')

print(f"Loaded {len(portuguese_stopwords)} Portuguese stopwords")
print("First 10 stopwords:", portuguese_stopwords[:10])

Loaded 207 Portuguese stopwords
First 10 stopwords: ['a', 'à', 'ao', 'aos', 'aquela', 'aquelas', 'aquele', 'aqueles', 'aquilo', 'as']


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


1. Separate offensive and non-offensive comments.

2. Tokenize (divide into words) each group.

3. Remove stop words.

4. Count the most frequent words in each group.

5. Calculate an “offensiveness score” for each word based on:

how many times it appears in offensive comments

how many times it appears in non-offensive comments

6. Keep the words that are very frequent in offensive comments and rare in non-offensive comments → your “unique offensive words.”

In [12]:
# Usamos la limpieza de text_clean_emb porque es una limpieza más exhaustiva y así podemos quedarnos solo con palabras y no tener un emeji como una de las palabras más comunes ofensivas
offensive_comments = df[df['label'] == 1]['text_clean_emb']
inoffensive_comments = df[df['label'] == 0]['text_clean_emb']

# Tokeniza los comentarios
all_offensive_tokens = [word for comment in offensive_comments for word in comment.split()]
all_inoffensive_tokens = [word for comment in inoffensive_comments for word in comment.split()]

print(f"Total offensive tokens: {len(all_offensive_tokens)}")
print(f"Total inoffensive tokens: {len(all_inoffensive_tokens)}")

# Creas nuevas listas eliminando las stopwords en portugués
filtered_offensive_tokens = [word for word in all_offensive_tokens if word not in portuguese_stopwords]
filtered_inoffensive_tokens = [word for word in all_inoffensive_tokens if word not in portuguese_stopwords]

print(f"Total offensive tokens after stopword removal: {len(filtered_offensive_tokens)}")
print(f"Total inoffensive tokens after stopword removal: {len(filtered_inoffensive_tokens)}")

# Usamos Counter y no len() porque tneemos una tupla (palabra, frecuencia)
from collections import Counter

offensive_word_counts = Counter(filtered_offensive_tokens)
inoffensive_word_counts = Counter(filtered_inoffensive_tokens)

print("Top 10 most common offensive words (after stopword removal):")
print(offensive_word_counts.most_common(10))
print("\nTop 10 most common inoffensive words (after stopword removal):")
print(inoffensive_word_counts.most_common(10))

unique_offensive_tokens_scored = {}

# Consider words present in offensive comments
for word, offensive_count in offensive_word_counts.items():
    inoffensive_count = inoffensive_word_counts.get(word, 0)      # Para cada una, miras su frecuencia en no ofensivos --> devuelve 0 sino aparece

    # Calculas un score de ofensividad:
    score = offensive_count / (inoffensive_count + 1) * offensive_count
# offensive_count / (inoffensive_count + 1):
# Si la palabra NO sale en no ofensivos (inoffensive_count = 0), es offensive_count / 1 → grande
# Si también aparece en no ofensivos, el denominador sube y el cociente baja
# Luego multiplicas por offensive_count otra vez → das más peso a palabras muy frecuentes en ofensivos

    # Filtro para quedarte solo con palabras realmente relevantes
    if offensive_count > 10 and score > 50:
        unique_offensive_tokens_scored[word] = score

# Ordenanos las (palabra, score) por el score (item[1]) de mayor a menor (reverse=True) --> Resultado: lista ordenada de las palabras ofensivas “más distintivas”
sorted_unique_offensive_tokens = sorted(unique_offensive_tokens_scored.items(), key=lambda item: item[1], reverse=True)

print("\nTop 20 Unique Offensive Tokens (highly frequent in offensive, rare in inoffensive):")
for word, score in sorted_unique_offensive_tokens[:20]:
    print(f"- {word} (Offensive Count: {offensive_word_counts[word]}, Inoffensive Count: {inoffensive_word_counts.get(word, 0)}, Score: {score:.2f})")

# Me quedo solo con las palabras (sin el score) y las meto en un set (sin duplicados)
unique_offensive_words = set(item[0] for item in sorted_unique_offensive_tokens)
print(f"Extracted {len(unique_offensive_words)} unique offensive words.")
print(f"First 10 unique offensive words: {list(unique_offensive_words)[:10]}")

Total offensive tokens: 53047
Total inoffensive tokens: 42523
Total offensive tokens after stopword removal: 31254
Total inoffensive tokens after stopword removal: 25031
Top 10 most common offensive words (after stopword removal):
[('pra', 357), ('vai', 270), ('brasil', 235), ('cara', 188), ('vc', 178), ('presidente', 163), ('povo', 156), ('vergonha', 155), ('esquerda', 144), ('tá', 138)]

Top 10 most common inoffensive words (after stopword removal):
[('presidente', 358), ('parabéns', 334), ('deus', 290), ('brasil', 266), ('vc', 175), ('pra', 171), ('bolsonaro', 167), ('sempre', 137), ('bem', 134), ('lula', 124)]

Top 20 Unique Offensive Tokens (highly frequent in offensive, rare in inoffensive):
- merda (Offensive Count: 104, Inoffensive Count: 0, Score: 10816.00)
- pirralha (Offensive Count: 134, Inoffensive Count: 1, Score: 8978.00)
- cadeia (Offensive Count: 91, Inoffensive Count: 0, Score: 8281.00)
- lixo (Offensive Count: 135, Inoffensive Count: 2, Score: 6075.00)
- nojo (Offens

In [13]:
def extract_linguistic_features(text):
    doc = nlp(text)
    features = {}

    # Básicas
    features['n_tokens'] = len(doc)
    features['upper_ratio'] = sum(1 for c in text if c.isupper()) / len(text) if text else 0
    features['n_exclam'] = text.count('!')
    features['n_question'] = text.count('?')

    # POS tags (solo adjetivos, sustantivos, verbos)  -> Son categorías especialmente relacionadas con lenguaje ofensivo
    pos_counts = {'ADJ': 0, 'NOUN': 0, 'VERB': 0}
    for token in doc:
        if token.pos_ in pos_counts:
            pos_counts[token.pos_] += 1         # si coincide la categoría gramatical con las que tenemos en pos_counts entonces sumas 1
    total = sum(pos_counts.values())
    for pos in pos_counts:
        features[f'prop_{pos.lower()}'] = pos_counts[pos] / total if total > 0 else 0

    # Emojis
    features['n_emojis'] = len(emoji.emoji_list(text))

    # Nuevas Features Solicitadas:
    # 1. Presencia de hashtags
    features['n_hashtags'] = len(re.findall(r'#\w+', text))                               # empiezan por # y siguen con caracteres de palabra (\w+)

    # 2. Presencia de menciones (@usuario)
    features['n_mentions'] = len(re.findall(r'@\w+', text))                       # Cuenta cuántas menciones a otros usuarios hay en el comentario

    # 3. Presencia de pronombres de segunda persona
    n_second_person_pronouns = 0
    for token in doc:
        # Check if it's a pronoun and has 'Person=2' morphology
        if token.pos_ == "PRON" and token.morph.get("Person") == ["2"]:
            n_second_person_pronouns += 1
    features['n_second_person_pronouns'] = n_second_person_pronouns

    # 4. Presencia del patrón NOUN + ADJ
    n_noun_adj_pattern = 0
    for i, token in enumerate(doc):
        if token.pos_ == "NOUN" and i + 1 < len(doc) and doc[i + 1].pos_ == "ADJ":                    # Existe un token siguiente (i + 1 < len(doc))
            n_noun_adj_pattern += 1
    features['n_noun_adj_pattern'] = n_noun_adj_pattern

    # 5. Presencia del patrón pronombre 2ª persona + adjetivo
    n_pron_2nd_adj_pattern = 0
    for i, token in enumerate(doc):
        if token.pos_ == "PRON" and token.morph.get("Person") == ["2"] and i + 1 < len(doc) and doc[i + 1].pos_ == "ADJ":
            n_pron_2nd_adj_pattern += 1
    features['n_pron_2nd_adj_pattern'] = n_pron_2nd_adj_pattern

    # New Feature: Count of unique offensive tokens
    n_unique_offensive_tokens = 0
    for token in doc:
        if token.lemma_.lower() in unique_offensive_words:                   # token.lemma_.lower() → lematizamos y pasamos a minúsculas
            n_unique_offensive_tokens += 1
    features['n_unique_offensive_tokens'] = n_unique_offensive_tokens


    return features

print("Extracting linguistic features...")
feat_list = df['text_features'].apply(extract_linguistic_features)
feat_df = pd.DataFrame(feat_list.tolist())
print(f"Features extracted: {feat_df.shape[1]}")


Extracting linguistic features...
Features extracted: 14


Pass the text through spaCy (nlp) → obtain tokens, POS, morphology, etc.
Extract:

  basic things (length, capital letters, !, ? , emojis),

  proportions of ADJ / NOUN / VERB --> IMPORTANT: These proportions are not relative to the total number of tokens, but relative to the total number of ADJ + NOUN + VERB --> That's why they always add up to 1

  number of hashtags,

  number of mentions,

  number of 2nd person pronouns,

  number of NOUN + ADJ patterns,

  number of 2nd person PRON + ADJ patterns,

  number of tokens belonging to your unique_offensive_words list.

  Returns a features dictionary with all of that.

In [14]:
print(feat_df.head())

   n_tokens  upper_ratio  n_exclam  n_question  prop_adj  prop_noun  \
0         3     0.083333         0           0      0.00       1.00   
1         8     0.032258         2           0      0.00       0.75   
2         5     0.033333         1           0      0.00       1.00   
3         3     0.052632         0           0      0.00       0.50   
4        15     0.051282         0           0      0.25       0.50   

   prop_verb  n_emojis  n_hashtags  n_mentions  n_second_person_pronouns  \
0       0.00         0           0           0                         0   
1       0.25         0           0           0                         0   
2       0.00         0           0           0                         0   
3       0.50         0           0           0                         0   
4       0.25         0           0           0                         0   

   n_noun_adj_pattern  n_pron_2nd_adj_pattern  n_unique_offensive_tokens  
0                   0                    

In [15]:
print(df['text'].head(5))

0                                         Mais um lixo
1                      Essa nao tem vergonha na cara!!
2                       Essa mulher é doente.pilantra!
3                                  Comunista safada...
4    Vagabunda. Comunista. Mentirosa. O povo chilen...
Name: text, dtype: object


5. DATA DIVISION

In [16]:
# ============================================================
# DIVISIÓN DE DATOS
# ============================================================

X_text_tfidf = df['text_clean_tfidf']
X_text_emb = df['text_clean_emb']
X_features = feat_df.values                                                    # matriz NumPy con tus features lingüísticas extraídas con spaCy
y = df['label'].values

# 1. Separar 30% temporal (val + test)
X_train_tfidf, X_temp_tfidf, \
X_train_emb, X_temp_emb, \
X_feat_train, X_temp_feat, \
y_train, y_temp = train_test_split(                                          # el comentario i en X_train_tfidf corresponde al mismo comentario i en: X_train_emb, X_feat_train, y_train
    X_text_tfidf, X_text_emb, X_features, y,                                  # estratificamos aunque en nuestro dataset las clses ya estan balanceadas --> buenas practicas
    test_size=0.3, random_state=42, stratify=y
)

# 2. Del 30% temporal → 15% val + 15% test
X_val_tfidf, X_test_tfidf, \
X_val_emb, X_test_emb, \
X_val_feat, X_feat_test, \
y_val, y_test = train_test_split(
    X_temp_tfidf, X_temp_emb, X_temp_feat, y_temp,
    test_size=0.5, random_state=42, stratify=y_temp
)

# Escalar solo con train
scaler = StandardScaler()
X_feat_train_scaled = scaler.fit_transform(X_feat_train)
X_val_feat_scaled = scaler.transform(X_val_feat)
X_feat_test_scaled = scaler.transform(X_feat_test)

# Escalado: necesario para combinar con TF-IDF y embeddings


Why do you only scale linguistic features and not TF-IDF or embeddings?

Because:

 TF-IDF is already normalized internally.

Vectorizers such as TfidfVectorizer generate matrices with already standardized values.

 Embeddings (GloVe) → each vector already has homogeneous values.

You should not rescale embeddings.

 Linguistic features → the only ones that require scaling.

Example:

No. of tokens ≈ 20

No. of emojis ≈ 0–2

Proportions ≈ 0–1

No. of NOUN+ADJ patterns ≈ 0–3

No. of hashtags ≈ 0–5

→ To prevent a classifier from giving more importance to a feature just because it has larger numbers, the scales must be homogenized.

6. MODEL A: TF-IDF + FEATURES + LogisticRegression

In [17]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score
from scipy.sparse import hstack, csr_matrix

# ============================================================
# MODELO A: TF-IDF (1-2 n-grams) + FEATURES LINGÜÍSTICAS
# ============================================================
print("\n" + "="*70)
print("TRAINING MODEL A: TF-IDF (1-2 n-grams) + LINGUISTIC FEATURES")
print("GridSearchCV with cv=3 ONLY in TRAIN")
print("Model selection using ACCURACY in VALIDATION")
print("TEST will be used only for the winning GLOBAL model")
print("="*70)

# Vectorizador TF-IDF
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),             # usa unigramas y bigramas → captura insultos compuestos (“vai embora”)
    lowercase=False,                   # mantienes mayúsculas (“IDIOTA” ≠ “idiota”)
    max_features=10000                    # límite del tamaño del vocabulario
)

X_tfidf_train = tfidf.fit_transform(X_train_tfidf)              # SOLO en train → aprende el vocabulario
X_tfidf_val   = tfidf.transform(X_val_tfidf)                     # aplica el vocabulario ya aprendido tanto en val como en test
X_tfidf_test  = tfidf.transform(X_test_tfidf)

# Combinar TF-IDF con features lingüísticas escaladas                                               # hstack concatena matrices por columnas
X_train_A = hstack([X_tfidf_train, csr_matrix(X_feat_train_scaled)])
X_val_A   = hstack([X_tfidf_val,   csr_matrix(X_val_feat_scaled)])
X_test_A  = hstack([X_tfidf_test,  csr_matrix(X_feat_test_scaled)])                     # SOLO se usará si el ganador global es de A
# TF-IDF → matriz dispersa (sparse matrix)
# Features lingüísticas → convertir a csr_matrix para que sean compatibles
# Se combinan en una sola matriz gigante de features

print(f"TF-IDF vocabulary size: {len(tfidf.vocabulary_):,} terms")
print(f"Train shape: {X_train_A.shape} | Val shape: {X_val_A.shape} | Test shape: {X_test_A.shape}")

# Diccionario para guardar resultados (SOLO VALIDATION)
results_A = {}

# ------------------------------------------------------------
# 1. Logistic Regression
# ------------------------------------------------------------
print("\nTraining Logistic Regression (GridSearch on C)...")
log_grid_A = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),                # lass_weight='balanced': reequilibra pesos si las clases están desbalanceadas aunque no es necesario en nuestro caso
    param_grid={'C': [0.1, 1.0, 10.0]},
    cv=3,                    # 3-fold CV solo en TRAIN
    scoring='accuracy',      # TUNING por accuracy
    n_jobs=-1                            # usa todos los núcleos disponibles
)
log_grid_A.fit(X_train_A, y_train)                # Ajusta GridSearch sobre el TRAIN

# Predicción en VALIDATION (accuray y f1_score)
y_val_pred_log = log_grid_A.predict(X_val_A)
acc_val_log = accuracy_score(y_val, y_val_pred_log)
f1_val_log  = f1_score(y_val, y_val_pred_log, average='macro')

results_A['LogisticRegression'] = {                               # guarda las métricas obtenidas y los mejores hiperparámetros
    'acc_val': acc_val_log,
    'f1_val': f1_val_log,
    'best_params': log_grid_A.best_params_,
    'best_estimator': log_grid_A.best_estimator_
}

print(f"   Best C: {log_grid_A.best_params_['C']} | "
      f"Acc (VAL): {acc_val_log:.4f} | F1 (VAL): {f1_val_log:.4f}")

# ------------------------------------------------------------
# 2. Random Forest
# ------------------------------------------------------------
print("Training Random Forest...")
rf_grid_A = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid={
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 20, 30]
    },
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)
rf_grid_A.fit(X_train_A, y_train)

y_val_pred_rf = rf_grid_A.predict(X_val_A)
acc_val_rf = accuracy_score(y_val, y_val_pred_rf)
f1_val_rf  = f1_score(y_val, y_val_pred_rf, average='macro')

results_A['RandomForest'] = {
    'acc_val': acc_val_rf,
    'f1_val': f1_val_rf,
    'best_params': rf_grid_A.best_params_,
    'best_estimator': rf_grid_A.best_estimator_
}

print(f"   Best params: {rf_grid_A.best_params_} | "
      f"Acc (VAL): {acc_val_rf:.4f} | F1 (VAL): {f1_val_rf:.4f}")

# ------------------------------------------------------------
# 3. Linear SVM
# ------------------------------------------------------------
print("Training Linear SVM...")
svm_grid_A = GridSearchCV(
    LinearSVC(class_weight='balanced', max_iter=10000, random_state=42),
    param_grid={'C': [0.01, 0.1, 1.0, 10.0]},
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)
svm_grid_A.fit(X_train_A, y_train)

y_val_pred_svm = svm_grid_A.predict(X_val_A)
acc_val_svm = accuracy_score(y_val, y_val_pred_svm)
f1_val_svm  = f1_score(y_val, y_val_pred_svm, average='macro')

results_A['SVM'] = {
    'acc_val': acc_val_svm,
    'f1_val': f1_val_svm,
    'best_params': svm_grid_A.best_params_,
    'best_estimator': svm_grid_A.best_estimator_
}

print(f"   Best C: {svm_grid_A.best_params_['C']} | "
      f"Acc (VAL): {acc_val_svm:.4f} | F1 (VAL): {f1_val_svm:.4f}")

# ------------------------------------------------------------
# RESUMEN MODELO A (USANDO VALIDATION)
# ------------------------------------------------------------
print("\n" + "="*60)
print("MODEL A SUMMARY (TF-IDF + Linguistic Features)")
print("="*60)
for name, info in results_A.items():
    print(f"{name:18} → Acc (VAL): {info['acc_val']:.4f} | "
          f"F1 (VAL): {info['f1_val']:.4f} | Best params: {info['best_params']}")
print("="*60)


TRAINING MODEL A: TF-IDF (1-2 n-grams) + LINGUISTIC FEATURES
GridSearchCV with cv=3 ONLY in TRAIN
Model selection using ACCURACY in VALIDATION
TEST will be used only for the winning GLOBAL model
TF-IDF vocabulary size: 10,000 terms
Train shape: (4900, 10014) | Val shape: (1050, 10014) | Test shape: (1050, 10014)

Training Logistic Regression (GridSearch on C)...
   Best C: 10.0 | Acc (VAL): 0.8267 | F1 (VAL): 0.8266
Training Random Forest...
   Best params: {'max_depth': None, 'n_estimators': 300} | Acc (VAL): 0.7695 | F1 (VAL): 0.7693
Training Linear SVM...
   Best C: 1.0 | Acc (VAL): 0.8219 | F1 (VAL): 0.8219

MODEL A SUMMARY (TF-IDF + Linguistic Features)
LogisticRegression → Acc (VAL): 0.8267 | F1 (VAL): 0.8266 | Best params: {'C': 10.0}
RandomForest       → Acc (VAL): 0.7695 | F1 (VAL): 0.7693 | Best params: {'max_depth': None, 'n_estimators': 300}
SVM                → Acc (VAL): 0.8219 | F1 (VAL): 0.8219 | Best params: {'C': 1.0}


7. MODEL B: GloVe + FEATURES + RandomForest

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score

# ============================================================
# MODELO B: GLOVE EMBEDDINGS + FEATURES
# ============================================================

print("\n" + "="*60)
print("TRAINING MODEL B: GLOVE + FEATURES")
print("="*60)

# Cargar GloVe (ajusta la ruta a la tuya)
glove_path = "/content/drive/MyDrive/proyecto2_PLN/glove_s100.txt"
print(f"Loading GloVe from: {glove_path}")
w2v_model = KeyedVectors.load_word2vec_format(glove_path)

def get_avg_embedding(text, model):
    words = text.split()
    vecs = []
    for w in words:
        if w in model:
            vecs.append(model[w])
    if not vecs:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

print("Calculating average embeddings for train, val, and test")
X_train_glove = np.vstack([get_avg_embedding(t, w2v_model) for t in X_train_emb])
X_val_glove   = np.vstack([get_avg_embedding(t, w2v_model) for t in X_val_emb])
X_test_glove  = np.vstack([get_avg_embedding(t, w2v_model) for t in X_test_emb])

print(f"GloVe embedding dimension: {w2v_model.vector_size}")
print(f"Train shape: {X_train_glove.shape} | Val shape: {X_val_glove.shape} | Test shape: {X_test_glove.shape}")

# Combinar embeddings con features lingüísticas
X_train_B = np.hstack([X_train_glove, X_feat_train_scaled])
X_val_B   = np.hstack([X_val_glove,   X_val_feat_scaled])
X_test_B  = np.hstack([X_test_glove,  X_feat_test_scaled])  # Solo se usará si el ganador global es de B

print(f"Train shape: {X_train_B.shape} | Val shape: {X_val_B.shape} | Test shape: {X_test_B.shape}")

# Diccionario para guardar resultados de B (SOLO VALIDATION)
results_B = {}

# ------------------------------------------------------------
# 1. Logistic Regression
# ------------------------------------------------------------
print("\nTraining Logistic Regression (GridSearch on C).")
log_grid_B = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    param_grid={'C': [0.1, 1.0, 10.0]},
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)
log_grid_B.fit(X_train_B, y_train)

y_val_pred_log_B = log_grid_B.predict(X_val_B)
acc_val_log_B = accuracy_score(y_val, y_val_pred_log_B)
f1_val_log_B  = f1_score(y_val, y_val_pred_log_B, average='macro')

results_B['LogisticRegression'] = {
    'acc_val': acc_val_log_B,
    'f1_val': f1_val_log_B,
    'best_params': log_grid_B.best_params_,
    'best_estimator': log_grid_B.best_estimator_
}

print(f"   Best C: {log_grid_B.best_params_['C']} | "
      f"Acc (VAL): {acc_val_log_B:.4f} | F1 (VAL): {f1_val_log_B:.4f}")

# ------------------------------------------------------------
# 2. Random Forest
# ------------------------------------------------------------
print("Training Random Forest.")
rf_grid_B = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid={
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 20, 30]
    },
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)
rf_grid_B.fit(X_train_B, y_train)

y_val_pred_rf_B = rf_grid_B.predict(X_val_B)
acc_val_rf_B = accuracy_score(y_val, y_val_pred_rf_B)
f1_val_rf_B  = f1_score(y_val, y_val_pred_rf_B, average='macro')

results_B['RandomForest'] = {
    'acc_val': acc_val_rf_B,
    'f1_val': f1_val_rf_B,
    'best_params': rf_grid_B.best_params_,
    'best_estimator': rf_grid_B.best_estimator_
}

print(f"   Best params: {rf_grid_B.best_params_} | "
      f"Acc (VAL): {acc_val_rf_B:.4f} | F1 (VAL): {f1_val_rf_B:.4f}")

# ------------------------------------------------------------
# 3. Linear SVM
# ------------------------------------------------------------
print("Training Linear SVM.")
svm_grid_B = GridSearchCV(
    LinearSVC(class_weight='balanced', max_iter=10000, random_state=42),
    param_grid={'C': [0.01, 0.1, 1.0, 10.0]},
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)
svm_grid_B.fit(X_train_B, y_train)

y_val_pred_svm_B = svm_grid_B.predict(X_val_B)
acc_val_svm_B = accuracy_score(y_val, y_val_pred_svm_B)
f1_val_svm_B  = f1_score(y_val, y_val_pred_svm_B, average='macro')

results_B['SVM'] = {
    'acc_val': acc_val_svm_B,
    'f1_val': f1_val_svm_B,
    'best_params': svm_grid_B.best_params_,
    'best_estimator': svm_grid_B.best_estimator_
}

print(f"   Best C: {svm_grid_B.best_params_['C']} | "
      f"Acc (VAL): {acc_val_svm_B:.4f} | F1 (VAL): {f1_val_svm_B:.4f}")

# ------------------------------------------------------------
# RESUMEN MODELO B (USANDO VALIDATION)
# ------------------------------------------------------------
print("\n" + "="*60)
print("MODEL B SUMMARY (GloVe avg + Linguistic Features)")
print("="*60)
for name, info in results_B.items():
    print(f"{name:18} → Acc (VAL): {info['acc_val']:.4f} | "
          f"F1 (VAL): {info['f1_val']:.4f} | Best params: {info['best_params']}")
print("="*60)



TRAINING MODEL B: GLOVE + FEATURES
Loading GloVe from: /content/drive/MyDrive/proyecto2_PLN/glove_s100.txt
Calculating average embeddings for train, val, and test
GloVe embedding dimension: 100
Train shape: (4900, 100) | Val shape: (1050, 100) | Test shape: (1050, 100)
Train shape: (4900, 114) | Val shape: (1050, 114) | Test shape: (1050, 114)

Training Logistic Regression (GridSearch on C).
   Best C: 1.0 | Acc (VAL): 0.8457 | F1 (VAL): 0.8457
Training Random Forest.
   Best params: {'max_depth': None, 'n_estimators': 300} | Acc (VAL): 0.8210 | F1 (VAL): 0.8208
Training Linear SVM.
   Best C: 0.1 | Acc (VAL): 0.8467 | F1 (VAL): 0.8466

MODEL B SUMMARY (GloVe avg + Linguistic Features)
LogisticRegression → Acc (VAL): 0.8457 | F1 (VAL): 0.8457 | Best params: {'C': 1.0}
RandomForest       → Acc (VAL): 0.8210 | F1 (VAL): 0.8208 | Best params: {'max_depth': None, 'n_estimators': 300}
SVM                → Acc (VAL): 0.8467 | F1 (VAL): 0.8466 | Best params: {'C': 0.1}


Again: there is no prediction in TEST yet. We just prepare everything and evaluate in VALIDATION.

8. Select GLOBAL WINNER and evaluate in TEST (only once)

In [21]:
from sklearn.metrics import classification_report, confusion_matrix

# ============================================================
# SELECCIÓN DEL GANADOR GLOBAL (A vs B) POR VALIDATION
# ============================================================

all_models = []

for name, info in results_A.items():
    all_models.append({
        'family': 'A',
        'model_name': name,
        'acc_val': info['acc_val'],
        'f1_val': info['f1_val'],
        'best_estimator': info['best_estimator']
    })

for name, info in results_B.items():
    all_models.append({
        'family': 'B',
        'model_name': name,
        'acc_val': info['acc_val'],
        'f1_val': info['f1_val'],
        'best_estimator': info['best_estimator']
    })

# Ordenar por Accuracy en VALIDATION (descendente)
all_models_sorted = sorted(all_models, key=lambda d: d['acc_val'], reverse=True)

print("\nMODEL RANKING ACCORDING TO VALIDATION:")
for m in all_models_sorted:
    print(f"{m['family']} - {m['model_name']:18} → Acc (VAL): {m['acc_val']:.4f} | F1 (VAL): {m['f1_val']:.4f}")

best_global = all_models_sorted[0]
print("\n" + "="*60)
print("BEST GLOBAL MODEL (according to VALIDATION)")
print("="*60)
print(f"Family: {best_global['family']}")
print(f"Model : {best_global['model_name']}")
print(f"Accuracy (VAL): {best_global['acc_val']:.4f}")
print(f"F1 (VAL)      : {best_global['f1_val']:.4f}")

# ============================================================
# EVALUACIÓN FINAL SOLO EN TEST DEL GANADOR GLOBAL
# ============================================================

if best_global['family'] == 'A':
    X_test_best = X_test_A
else:
    X_test_best = X_test_B

best_estimator = best_global['best_estimator']

y_test_pred = best_estimator.predict(X_test_best)
acc_test = accuracy_score(y_test, y_test_pred)
f1_test  = f1_score(y_test, y_test_pred, average='macro')

print("\n" + "="*60)
print("FINAL TEST RESULTS (OVERALL WINNER ONLY)")
print("="*60)
print(f"Accuracy (TEST): {acc_test:.4f}")
print(f"F1-macro (TEST): {f1_test:.4f}")
print("\nClassification report (TEST):")
print(classification_report(y_test, y_test_pred, digits=4))
print("Confusion matrix (TEST):")
print(confusion_matrix(y_test, y_test_pred))
print("="*60)


MODEL RANKING ACCORDING TO VALIDATION:
B - SVM                → Acc (VAL): 0.8467 | F1 (VAL): 0.8466
B - LogisticRegression → Acc (VAL): 0.8457 | F1 (VAL): 0.8457
A - LogisticRegression → Acc (VAL): 0.8267 | F1 (VAL): 0.8266
A - SVM                → Acc (VAL): 0.8219 | F1 (VAL): 0.8219
B - RandomForest       → Acc (VAL): 0.8210 | F1 (VAL): 0.8208
A - RandomForest       → Acc (VAL): 0.7695 | F1 (VAL): 0.7693

BEST GLOBAL MODEL (according to VALIDATION)
Family: B
Model : SVM
Accuracy (VAL): 0.8467
F1 (VAL)      : 0.8466

FINAL TEST RESULTS (OVERALL WINNER ONLY)
Accuracy (TEST): 0.8248
F1-macro (TEST): 0.8248

Classification report (TEST):
              precision    recall  f1-score   support

           0     0.8211    0.8305    0.8258       525
           1     0.8285    0.8190    0.8238       525

    accuracy                         0.8248      1050
   macro avg     0.8248    0.8248    0.8248      1050
weighted avg     0.8248    0.8248    0.8248      1050

Confusion matrix (TEST):
[[

Compare the 6 models by accuracy in VALIDATION.

Choose the best one.

Evaluate it in TEST (for the first and only time).